# cfos-mapper from the command line

Every step of the analysis is one `cfos-mapper` command. In this notebook the commands run with `!`
(a terminal command inside Jupyter); in a terminal, type them without the `!`.

Long steps (training, whole brains, registration) should run in `tmux` on the GPU machine, or as cluster
jobs (end of this notebook). The napari plugin starts the same commands in the background.

In [ ]:
!cfos-mapper --help

## 1. A project

`cfos-mapper init FOLDER` writes a template `settings.json` and `regions.json`. Edit `brains_folder`
(one `<brain>.ome.zarr` per brain) and `results_folder`.

In [ ]:
!cfos-mapper init my_project
!head -n 12 my_project/settings.json

Commands find the settings in `--settings FILE`, else in the environment variable `CFOS_MAPPER_SETTINGS`,
else in `./settings.json`. Here we use our own project:

In [ ]:
import os

# the settings file of the project (where the brains are, where results go, all parameters)
os.environ["CFOS_MAPPER_SETTINGS"] = "/mif-users/no_backup/Gritti_Nicola/IDIBAPS-brains/code/settings.json"

## 2. Test regions and their candidates

Test regions are small boxes of a brain (`regions.json`, usually drawn in napari, widget 1).
`prepare` cuts one out with pymif and finds its candidates (bright, cell-sized spots) with cellfinder.
`prepare-all` does it for every region that is not prepared yet.

In [ ]:
!cfos-mapper prepare-all

## 3. Annotate (napari)

Annotating is done by eye in napari: `cfos-mapper napari`, widget *2. Annotate and train*
(see `03_napari_plugin.ipynb`).

## 4. How good is a network on the test regions?

`predict` runs a network (`pretrained` = cellfinder's own, or one of ours in the networks folder) on a test
region and on random spots of plain tissue (a control: a good network calls them *not a cell*).

In [ ]:
!cfos-mapper predict O16 regionA_cortex pretrained
!cfos-mapper predict-all pretrained

## 5. Train a network on all annotations

Starts from cellfinder's pretrained network; 20% of the annotations are kept aside as a test set.
Results: `networks/<name>/test_results.png`. About 30 min for ~800 annotations and 150 epochs.

In [ ]:
# !cfos-mapper train network_v3 --epochs 150 --gpu 1

## 6. Whole brain

First a quick test on 100 planes (~10 min), then the whole brain (several hours, ~550 GB of memory).

In [ ]:
# !cfos-mapper run-brain O15 --network network_v3 --gpu 0 --planes 900 1000
# !cfos-mapper run-brain O15 --network network_v3 --gpu 0

## 7. Register to the atlas, count cells per region

`register` runs brainreg on the autofluorescence (CPU, ~30-60 min, once per brain). `count` places every cell
of a whole-brain run in its atlas region. `summary` puts all brains in one table.

In [ ]:
!cfos-mapper register O15

In [ ]:
!cfos-mapper count O15 --network network_v2
!cfos-mapper summary --network network_v2

In [ ]:
import pandas as pd
import cfos_mapper as cf

settings = cf.load_settings()
table = pd.read_csv(f"{settings['reports_folder']}/network_v2/cells_per_region_all_brains.csv")
table[table.depth <= 3]

## 8. On an HPC cluster (container)

Every release has a Docker image without GUI. With Apptainer (Singularity), `--nv` gives the container the GPU
and `--bind` the data folder. An example SLURM job:

In [ ]:
%%writefile run_O15.sbatch
#!/bin/bash
#SBATCH --job-name=cfos-O15
#SBATCH --gres=gpu:1
#SBATCH --mem=600G
#SBATCH --cpus-per-task=32
#SBATCH --time=24:00:00

apptainer exec --nv --bind /mif-users cfos-mapper.sif \
    cfos-mapper --settings /mif-users/no_backup/Gritti_Nicola/IDIBAPS-brains/code/settings.json \
    run-brain O15 --network network_v3

In [ ]:
# once: !apptainer pull cfos-mapper.sif docker://ghcr.io/embl-mif/cfos-mapper:latest
# then:  !sbatch run_O15.sbatch